# 01 — Integración y limpieza: películas + series
**Curso:** ADY1104 Visualización de Datos · **Evaluación:** EP3 · **Caso:** StreamView Analytics

**Objetivo de la fase (indicador IE16, integración de múltiples fuentes):** unir `netflix_movies_detailed_up_to_2025.csv` y `netflix_tv_shows_detailed_up_to_2025.csv` en un dataset consistente, documentando cada decisión.

La lógica vive en `src/limpieza.py` (reutilizable por el dashboard); este notebook la ejecuta y muestra la evidencia.

In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.append(str(Path.cwd().parent / 'src'))
import limpieza as lp
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)

peliculas, series = lp.cargar_crudos()
print('Películas:', peliculas.shape, '| Series:', series.shape)
print('Columnas solo en películas:', set(peliculas.columns) - set(series.columns))
print('Columnas solo en series   :', set(series.columns) - set(peliculas.columns))

Películas: (16000, 18) | Series: (16000, 16)
Columnas solo en películas: {'budget', 'revenue'}
Columnas solo en series   : set()


## 1. Diagnóstico comparado de las dos fuentes

In [2]:
def diag(d):
    return pd.DataFrame({'% nulos': (d.isna().mean()*100).round(1), 'únicos': d.nunique()})
pd.concat({'Películas': diag(peliculas), 'Series': diag(series)}, axis=1)

Películas         Series         
               % nulos únicos % nulos   únicos
show_id            0.0  16000     0.0  15991.0
type               0.0      1     0.0      1.0
title              0.0  15485     0.0  15668.0
director           0.8   9508    68.5   3808.0
cast               1.3  15639     7.2  14628.0
country            2.9   1463    11.2    564.0
date_added         0.0   4423     0.0   4899.0
release_year       0.0     16     0.0     16.0
rating             0.0   2145     0.0   1184.0
duration         100.0      0     0.0      1.0
genres             0.7   2768     6.1   1218.0
language           0.0     74     0.0     71.0
description        0.8  15854    20.0  12782.0
popularity         0.0  11173     0.0  14328.0
vote_count         0.0   2693     0.0   1021.0
vote_average       0.0   2145     0.0   1184.0
budget             0.0    958     NaN      NaN
revenue            0.0   5327     NaN      NaN

**Hallazgos que obligan a decidir antes de unir:**
- `duration`: 100 % vacía en películas y constante (“1 Seasons”) en series → se elimina.
- `rating` es idéntica a `vote_average` en ambas fuentes → se elimina.
- `director` nulo en 68,5 % de las series (0,8 % en películas): no sirve para comparar entre tipos.
- `budget` y `revenue` existen solo en películas.

In [3]:
solapados = set(peliculas.show_id) & set(series.show_id)
print('show_id que aparecen en ambos archivos:', len(solapados))
print('show_id repetidos dentro de series:', series.show_id.duplicated().sum())
ej = sorted(solapados)[:1][0]
pd.concat([peliculas[peliculas.show_id==ej][['show_id','title','release_year']].assign(archivo='películas'),
           series[series.show_id==ej][['show_id','title','release_year']].assign(archivo='series')])

show_id que aparecen en ambos archivos: 397
show_id repetidos dentro de series: 9


,show_id,title,release_year,archivo
24,18823,Clash of the Titans,2010,películas
485,18823,Growing Through Life,2010,series


**Decisión 1 — llave compuesta.** Los `show_id` son espacios distintos (el mismo número identifica títulos diferentes). Se crea `content_id` = `M-<show_id>` (película) o `S-<show_id>` (serie). Unir solo por `show_id` mezclaría títulos que no tienen relación.

**Decisión 2 — duplicados en series.** Los 9 `show_id` repetidos son de 2025; se conserva la fila con más votos y, si empatan, la de mayor popularidad.

In [4]:
series_ok, n_dups = lp.resolver_duplicados_series(series)
print('Series duplicadas eliminadas:', n_dups, '→ series finales:', len(series_ok))
df_integrado = lp.integrar(peliculas, series_ok)
print('Filas del dataset integrado:', len(df_integrado), '(esperado 16.000 + 15.991)')

Series duplicadas eliminadas: 9 → series finales: 15991
Filas del dataset integrado: 31991 (esperado 16.000 + 15.991)


## 2. Calificaciones: el cero no es una nota
En EP1 los `vote_average = 0` entraron en los promedios. Un cero con `vote_count = 0` significa **sin votos**, no “muy mal calificado”.

**Decisión 3.** Se crea `vote_average_clean` (NaN si `vote_count = 0`), la bandera `tiene_votos` y `calificacion_confiable` (`vote_count >= 10`).

In [5]:
df = lp.limpiar(df_integrado)
res = df.groupby('tipo_contenido').agg(
    titulos=('content_id', 'count'),
    sin_votos=('tiene_votos', lambda x: int((~x).sum())),
    con_votos_suficientes=('calificacion_confiable', 'sum'),
    calif_prom_EP1_con_ceros=('vote_average', 'mean'),
    calif_prom_corregida=('vote_average_clean', 'mean'))
res['% con votos suficientes'] = (res['con_votos_suficientes'] / res['titulos'] * 100).round(1)
res.round(3)

,titulos,sin_votos,con_votos_suficientes,calif_prom_EP1_con_ceros,calif_prom_corregida,% con votos suficientes
tipo_contenido,,,,,,
Película,16000,894,14183,5.956,6.309,88.6
Serie,15991,3666,6148,5.420,7.024,38.4


Las series quedan con 7,02 (antes 5,42) y las películas con 6,31 (antes 5,96). Solo 38,4 % de las series tiene votos suficientes: **toda comparación de calidad debe mostrar esa cobertura**.

**Decisión 4 — popularidad.** La escala de `popularity` difiere por tipo (mediana 10,9 en películas vs 36,2 en series). Para comparar se crea `indice_engagement`: percentil (0–100) de la popularidad **dentro de su tipo**. La popularidad cruda se conserva para análisis dentro de un mismo tipo.

In [6]:
df.groupby('tipo_contenido').agg(popularidad_mediana=('popularity','median'),
                                 popularidad_promedio=('popularity','mean'),
                                 indice_engagement_mediano=('indice_engagement','median')).round(2)

,popularidad_mediana,popularidad_promedio,indice_engagement_mediano
tipo_contenido,,,
Película,10.91,20.38,50.00
Serie,36.20,64.88,50.01


## 3. Finanzas (solo películas)
`budget`/`revenue` = 0 se interpretan como dato no informado (igual que EP1). Para series quedan en NaN: ROI y directores por ingresos **solo aplican a películas**.

In [7]:
fin = df.groupby('tipo_contenido').agg(titulos=('content_id','count'),
                                       con_datos_financieros=('tiene_datos_financieros','sum'))
fin['% del tipo'] = (fin['con_datos_financieros']/fin['titulos']*100).round(1)
print(fin)
print('ROI mediano (películas con datos):', round(df['roi'].median(), 2), '| ROI promedio:', round(df['roi'].mean(), 1))

                titulos  con_datos_financieros  % del tipo
tipo_contenido                                            
Película          16000                   3540        22.1
Serie             15991                      0         0.0
ROI mediano (películas con datos): 1.7 | ROI promedio: 781.5


## 4. Homologación de géneros
Las dos fuentes usan taxonomías distintas (películas: Action, Adventure, Science Fiction…; series: Action & Adventure, Sci-Fi & Fantasy, Kids, Soap, Talk…). Se define `mapa_generos.csv` y se trabaja con `genero_unificado`.

**Decisión 5.** Action y Adventure (películas) se unen en “Acción y Aventura” para ser equivalentes a “Action & Adventure” (series); lo mismo con ciencia ficción y fantasía. Al unir, se **eliminan repeticiones** de un mismo título dentro del mismo género unificado, para no contarlo dos veces.

In [8]:
gen, pais, direc = lp.tabla_genero(df), lp.tabla_pais(df), lp.tabla_director(df)
cuenta = gen.groupby(['genero_unificado','tipo_contenido']).size().unstack(fill_value=0)
cuenta['total'] = cuenta.sum(axis=1)
print('Géneros presentes en ambos tipos:', int(((cuenta['Película']>0)&(cuenta['Serie']>0)).sum()),
      '| solo películas:', int(((cuenta['Película']>0)&(cuenta['Serie']==0)).sum()),
      '| solo series:', int(((cuenta['Película']==0)&(cuenta['Serie']>0)).sum()))
cuenta.sort_values('total', ascending=False)

Géneros presentes en ambos tipos:

 12 | solo películas: 5 | solo series: 4


tipo_contenido,Película,Serie,total
genero_unificado,,,
Drama,6910,7859,14769
Comedia,4533,4572,9105
Acción y Aventura,4157,1988,6145
Ciencia ficción y Fantasía,2730,1958,4688
Animación,1579,2490,4069
Suspenso,3769,0,3769
Familia,1472,2082,3554
Crimen,1738,1465,3203
Misterio,1242,1332,2574


**Implicancia para los gráficos:** Suspenso, Terror, Romance, Historia, Música y Película de TV existen solo en películas; Reality, Telenovela, Talk show y Noticias solo en series. Las comparaciones directas entre tipos deben limitarse a los géneros presentes en ambos, y esto debe declararse en el gráfico.

## 5. Ejecución completa, controles de calidad y exportación

## 5. Exportación del pipeline y control de calidad
Se ejecutan todas las transformaciones integradas, se validan los 15 controles de calidad automáticos y se exportan las tablas procesadas a `data/processed/`.

In [9]:
df, gen, pais, direc, mapa, qc = lp.ejecutar(guardar=True)
qc

,control,esperado,obtenido,ok
0,Películas de entrada,16000,16000,True
1,Series de entrada,16000,16000,True
2,Series duplicadas eliminadas (show_id repetido),9,9,True
3,Filas del dataset unificado = películas + seri...,31991,31991,True
4,content_id único (duplicados),0,0,True
5,show_id que se solapan entre películas y serie...,397,397,True
6,Nulos en content_id/tipo_contenido/release_year,0,0,True
7,Series con budget/revenue (deben ser 0),0,0,True
8,Películas con datos financieros completos,3540,3540,True
9,vote_average_clean nulo solo si vote_count = 0...,0,0,True


In [10]:
assert qc['ok'].all(), 'Hay controles de calidad que fallan'
print('Todos los controles de calidad se cumplen.')
print('Archivos exportados a data/processed/:', sorted(p.name for p in lp.PROC.glob('*.csv')))

Todos los controles de calidad se cumplen.
Archivos exportados a data/processed/: ['cifras_clave_eda.csv', 'contenido_por_director.csv', 'contenido_por_genero.csv', 'contenido_por_pais.csv', 'contenido_unificado.csv', 'control_calidad.csv', 'control_g4_vs_looker.csv', 'kpis_por_anio.csv', 'kpis_por_genero.csv', 'kpis_por_pais.csv', 'kpis_resumen.csv', 'kpis_verificacion.csv', 'looker_contenido.csv', 'looker_genero.csv', 'mapa_generos.csv']


## 5b. Diccionario de variables y tablas derivadas
Variables del dataset integrado (`contenido_unificado.csv`) y su rol en el análisis, más las tablas largas que alimentan los gráficos de desglose.

In [11]:
DICCIONARIO = {
    "content_id": ("identificador", "Llave compuesta M-<show_id> / S-<show_id>"),
    "show_id": ("identificador", "ID original de TMDb (se repite entre archivos)"),
    "tipo_contenido": ("filtro", "Película o Serie"),
    "title": ("identificador", "Título"),
    "release_year": ("filtro", "Año de estreno (2010–2025)"),
    "date_added": ("origen", "Fecha de incorporación (se convierte a fecha)"),
    "genres": ("origen", "Géneros originales (taxonomías distintas por archivo)"),
    "country": ("filtro", "Países de producción"),
    "language": ("origen", "Código de idioma original"),
    "idioma": ("filtro", "Idioma original en español"),
    "director": ("filtro", "Director(es); muy incompleto en series"),
    "popularity": ("métrica", "Popularidad cruda (escala distinta por tipo)"),
    "vote_average": ("origen", "Nota original; incluye ceros sin votos"),
    "vote_count": ("métrica", "Cantidad de votos"),
    "vote_average_clean": ("métrica", "Nota con NaN si vote_count = 0"),
    "tiene_votos": ("derivada", "vote_count > 0"),
    "calificacion_confiable": ("derivada", "vote_count >= 10 (UMBRAL_VOTOS)"),
    "indice_engagement": ("métrica", "Percentil 0–100 de popularity dentro del tipo"),
    "budget": ("origen", "Presupuesto original (solo películas)"),
    "revenue": ("origen", "Ingresos originales (solo películas)"),
    "budget_clean": ("métrica", "Presupuesto; 0 = no informado (NaN)"),
    "revenue_clean": ("métrica", "Ingresos; 0 = no informado (NaN)"),
    "tiene_datos_financieros": ("derivada", "Presupuesto e ingresos válidos"),
    "roi": ("métrica", "revenue_clean / budget_clean"),
    "description": ("texto", "Sinopsis"),
    "cast": ("texto", "Reparto"),
}
dic = pd.DataFrame({
    "variable": df.columns,
    "rol": [DICCIONARIO.get(c, ("(completar)", ""))[0] for c in df.columns],
    "descripción": [DICCIONARIO.get(c, ("", "(completar)"))[1] for c in df.columns],
    "tipo de dato": [str(t) for t in df.dtypes],
    "% sin dato": ((df.isna() | df.eq("No especificado")).mean() * 100).round(1).values,
})
faltan = dic.loc[dic["rol"] == "(completar)", "variable"].tolist()
if faltan:
    print("Variables sin descripción (completar en DICCIONARIO):", faltan)
display(dic)

tablas = pd.DataFrame({
    "tabla": ["contenido_por_genero", "contenido_por_pais", "contenido_por_director"],
    "filas": [len(gen), len(pais), len(direc)],
    "títulos distintos": [gen.content_id.nunique(), pais.content_id.nunique(), direc.content_id.nunique()],
    "para qué sirve": ["Desglose por género unificado (G3, G4)",
                       "Desglose por país (G6)",
                       "ROI por director (G8)"],
})
display(tablas)

,variable,rol,descripción,tipo de dato,% sin dato
0,content_id,identificador,Llave compuesta M-<show_id> / S-<show_id>,str,0.0
1,show_id,identificador,ID original de TMDb (se repite entre archivos),int64,0.0
2,title,identificador,Título,str,0.0
3,director,filtro,Director(es); muy incompleto en series,str,34.7
4,cast,texto,Reparto,str,4.3
5,country,filtro,Países de producción,str,7.1
6,date_added,origen,Fecha de incorporación (se convierte a fecha),datetime64[us],0.0
7,release_year,filtro,Año de estreno (2010–2025),int64,0.0
8,genres,origen,Géneros originales (taxonomías distintas por a...,str,3.4
9,language,origen,Código de idioma original,str,0.0


,tabla,filas,títulos distintos,para qué sirve
0,contenido_por_genero,65021,31991,"Desglose por género unificado (G3, G4)"
1,contenido_por_pais,39889,31991,Desglose por país (G6)
2,contenido_por_director,24763,20899,ROI por director (G8)


## 6. Resumen de decisiones (para el informe)
| # | Decisión | Motivo |
|---|---|---|
| 1 | `content_id` = tipo + `show_id` | 397 `show_id` se solapan entre archivos |
| 2 | Conservar una fila por `show_id` duplicado en series | 9 repetidos (2025) |
| 3 | `vote_average_clean` y `calificacion_confiable` (≥ 10 votos) | El 0 sin votos distorsionaba los promedios de EP1 |
| 4 | `indice_engagement` (percentil por tipo) | Escalas de popularidad no comparables |
| 5 | `genero_unificado` con mapa explícito y sin repeticiones | Taxonomías distintas |
| 6 | Finanzas solo en películas | Las series no traen `budget` ni `revenue` |

**Límite declarado:** ambas fuentes tienen exactamente 1.000 títulos por año (muestra balanceada artificialmente), por lo que la composición por año no refleja el catálogo real.